# Agents, Part 1: build a tool-calling agent by hand (about 60-90 minutes)

**Goal:** watch an LLM decide, step by step, which tool to use, and see your code run it.

**What you build:** an agent that answers questions about a small **made-up claims table** using two tools:
- `run_sql`: runs a read-only SQL query on the table
- `calculator`: does arithmetic

**The loop you will write (think → act → observe → repeat):**

Question → model decides → (tool request → **your code runs the tool** → result back to the model) × N → final answer

**Remember the key fact:** the model never runs anything. It only *asks*. Your code decides whether and how to run the request.

**Cost:** under 10 cents with `gpt-4o-mini`. Uses your `OPENAI_API_KEY` from Colab Secrets, as in earlier steps. No Google Drive needed.


## 1. Setup

In [1]:
%pip install -q -U openai
import os, sys, json, ast, operator, sqlite3, random
import pandas as pd
from openai import OpenAI

if "google.colab" in sys.modules:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI()
MODEL = "gpt-4o-mini"
print("Ready")

Ready


## 2. Create a small made-up claims database

A SQLite database in memory with 300 synthetic claims. Look at the table so you know what questions make sense.

In [2]:
random.seed(42)
states = ["TX", "CA", "FL", "NY", "IL"]
types = ["auto", "home", "roadside"]
rows = []
for i in range(1, 301):
    t = random.choice(types)
    base = {"auto": 4000, "home": 9000, "roadside": 150}[t]
    rows.append({
        "claim_id": i,
        "state": random.choice(states),
        "claim_type": t,
        "cost_usd": round(base * random.uniform(0.3, 2.5), 2),
        "has_attorney": int(random.random() < (0.25 if t == "auto" else 0.08)),
        "report_month": random.choice(["2026-08", "2026-09", "2026-10"]),
    })
df = pd.DataFrame(rows)
conn = sqlite3.connect(":memory:")
df.to_sql("claims", conn, index=False)

SCHEMA = '''Table claims(
  claim_id INTEGER,
  state TEXT,          -- two-letter US state: TX, CA, FL, NY, IL
  claim_type TEXT,     -- auto, home or roadside
  cost_usd REAL,       -- claim cost in US dollars
  has_attorney INTEGER,-- 1 if an attorney is involved, else 0
  report_month TEXT    -- YYYY-MM, from 2026-08 to 2026-10
)'''
df.head()

,claim_id,state,claim_type,cost_usd,has_attorney,report_month
0,1,TX,roadside,53.25,0,2026-08
1,2,TX,auto,7154.96,0,2026-08
2,3,NY,roadside,55.49,0,2026-08
3,4,IL,roadside,53.76,0,2026-10
4,5,IL,roadside,183.44,0,2026-09


## 3. Write the tools (ordinary Python functions)

These are normal functions. Notice the **guardrails live here, in your code**, not in the prompt:
- `run_sql` allows only SELECT queries, adds a row LIMIT, and returns errors as text (so the model can read the error and fix its query).
- `calculator` evaluates arithmetic safely (no `eval`, which could run any code).

In [3]:
def run_sql(query: str) -> str:
    q = query.strip().rstrip(";")
    if not q.lower().startswith(("select", "with")) or ";" in q:
        return "ERROR: only a single SELECT query is allowed."
    if " limit " not in q.lower():
        q += " LIMIT 50"
    try:
        out = pd.read_sql_query(q, conn)
        return out.to_csv(index=False)
    except Exception as e:
        return f"ERROR: {e}"

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}
def calculator(expression: str) -> str:
    def ev(n):
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](ev(n.operand))
        raise ValueError("unsupported expression")
    try:
        return str(round(ev(ast.parse(expression, mode="eval").body), 4))
    except Exception as e:
        return f"ERROR: {e}"

# Test the tools yourself before giving them to the model
print(run_sql("SELECT state, COUNT(*) AS n FROM claims GROUP BY state"))
print(run_sql("DELETE FROM claims"))          # blocked by the guardrail
print(calculator("4200 * 83.5"))

state,n
CA,60
FL,61
IL,66
NY,61
TX,52

ERROR: only a single SELECT query is allowed.
350700.0


## 4. Describe the tools to the model

The model never sees your Python code. It only sees this **description**: a name, what the tool does, and the arguments as a JSON schema. Good descriptions matter as much as good prompts. Notice the schema is in the `run_sql` description, so the model knows the column names.

In [4]:
TOOLS = [
    {"type": "function", "function": {
        "name": "run_sql",
        "description": "Run one read-only SQLite SELECT query on the claims table and get CSV back.\n" + SCHEMA,
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string", "description": "A single SQLite SELECT query"}},
                       "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate an arithmetic expression such as '4200 * 83.5' or '(12 / 300) * 100'.",
        "parameters": {"type": "object",
                       "properties": {"expression": {"type": "string"}},
                       "required": ["expression"]}}},
]
TOOL_FUNCS = {"run_sql": run_sql, "calculator": calculator}

SYSTEM = ("You are a claims data assistant. Use the tools to get facts; never guess numbers. "
          "Use calculator for arithmetic. If the question cannot be answered from the claims table, say so. "
          "Keep the final answer short and state the numbers you found.")

## 5. One step only: see what a tool request looks like

Before the loop, make a **single** call and print what comes back. The model does not answer; it returns a `tool_calls` request with the tool name and arguments as JSON. Nothing has run yet.

In [5]:
messages = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": "What is the average auto claim cost in TX?"}]
r = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, temperature=0)
msg = r.choices[0].message
print("Text answer:", msg.content)
for call in msg.tool_calls or []:
    print("Tool requested:", call.function.name)
    print("Arguments     :", call.function.arguments)

Text answer: None
Tool requested: run_sql
Arguments     : {"query":"SELECT AVG(cost_usd) as average_auto_cost FROM claims WHERE state = 'TX' AND claim_type = 'auto'"}


**What you should see:** `Text answer: None` and a request to `run_sql` with a query the model wrote. Now we need code that runs it and sends the result back. That code, repeated, is the agent.

## 6. The agent loop

Read this cell slowly; it is the whole idea:
1. Call the model with the conversation so far and the tool list.
2. If it gives a plain answer (no tool calls), we are done.
3. Otherwise, run each requested tool **in our code**, append the result as a `tool` message, and go round again.
4. Stop after `max_steps` no matter what (the step cap).

In [6]:
def run_agent(question, max_steps=6, verbose=True):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        r = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, temperature=0)
        msg = r.choices[0].message

        if not msg.tool_calls:                                   # THINK: model decided it is done
            if verbose: print(f"\n[step {step}] FINAL ANSWER:\n{msg.content}")
            return msg.content

        # keep the model's request in the history (this is the agent's short-term memory)
        messages.append({"role": "assistant", "content": msg.content,
                         "tool_calls": [{"id": c.id, "type": "function",
                                         "function": {"name": c.function.name, "arguments": c.function.arguments}}
                                        for c in msg.tool_calls]})
        for c in msg.tool_calls:                                 # ACT: our code runs the tool
            args = json.loads(c.function.arguments)
            result = TOOL_FUNCS[c.function.name](**args)
            if verbose:
                print(f"\n[step {step}] model asked for {c.function.name}({args})")
                print("  result:", result[:300].replace("\n", " | "))
            messages.append({"role": "tool", "tool_call_id": c.id, "content": result})   # OBSERVE
    return "Stopped: hit the step limit without a final answer."

run_agent("What is the average auto claim cost in TX?")


[step 1] model asked for run_sql({'query': "SELECT AVG(cost_usd) as average_auto_cost FROM claims WHERE state = 'TX' AND claim_type = 'auto'"})
  result: average_auto_cost | 5808.775000000001 | 

[step 2] FINAL ANSWER:
The average auto claim cost in TX is $5808.78.


'The average auto claim cost in TX is $5808.78.'

## 7. Try questions that need 0, 1, 2 or more steps

Run each and **read the printed trace**. For each one, ask yourself: why did the model choose that tool, and when did it decide to stop?

In [7]:
questions = [
    "What is 15% of 2,40,000?",                                                   # calculator only
    "Which state has the most home claims?",                                      # one SQL query
    "What share of auto claims involve an attorney, as a percentage?",            # SQL, maybe calculator
    "What is the average TX auto claim cost in Indian rupees at 83.5 rupees per dollar?",  # SQL then calculator
    "Compare average claim cost with and without an attorney for auto claims. How many times higher is it?",
    "What will the claim cost be next year?",                                     # out of scope: should say it can't
]
for q in questions:
    print("=" * 100, "\nQUESTION:", q)
    run_agent(q)

QUESTION: What is 15% of 2,40,000?

[step 1] model asked for calculator({'expression': '0.15 * 240000'})
  result: 36000.0

[step 2] FINAL ANSWER:
15% of 240,000 is 36,000.
QUESTION: Which state has the most home claims?

[step 1] model asked for run_sql({'query': "SELECT state, COUNT(*) as home_claim_count FROM claims WHERE claim_type = 'home' GROUP BY state ORDER BY home_claim_count DESC LIMIT 1;"})
  result: state,home_claim_count | IL,26 | 

[step 2] FINAL ANSWER:
The state with the most home claims is Illinois (IL) with 26 claims.
QUESTION: What share of auto claims involve an attorney, as a percentage?

[step 1] model asked for run_sql({'query': "SELECT COUNT(*) as total_claims, SUM(has_attorney) as claims_with_attorney FROM claims WHERE claim_type = 'auto'"})
  result: total_claims,claims_with_attorney | 104,23 | 

[step 2] model asked for calculator({'expression': '(23 / 104) * 100'})
  result: 22.1154

[step 3] FINAL ANSWER:
Approximately 22.12% of auto claims involve an attor

## 8. Guardrails and self-correction

Two things to observe:
1. **The model can recover from errors.** Ask about a column that does not exist. The SQL tool returns an error message, the model reads it, and usually fixes its query on the next step. This is why tools return errors as text instead of crashing.
2. **The guardrail is in code.** Even if a user asks the agent to delete data, `run_sql` refuses. Never rely on the prompt alone for safety.

In [8]:
print("=" * 100); run_agent("What is the average cost of claims filed in Pune?")
print("=" * 100); run_agent("Delete all the roadside claims from the table.")
print("=" * 100); run_agent("Break down auto claims by state and month, then tell me which month grew fastest in TX.", max_steps=3)


[step 1] FINAL ANSWER:
I cannot answer that question as the claims data does not include information about Pune.

[step 1] FINAL ANSWER:
I cannot delete records from the claims table. I can only retrieve data.

[step 1] model asked for run_sql({'query': "SELECT state, report_month, COUNT(*) as claim_count FROM claims WHERE claim_type = 'auto' GROUP BY state, report_month ORDER BY state, report_month;"})
  result: state,report_month,claim_count | CA,2026-08,6 | CA,2026-09,7 | CA,2026-10,9 | FL,2026-08,5 | FL,2026-09,5 | FL,2026-10,11 | IL,2026-08,3 | IL,2026-09,12 | IL,2026-10,7 | NY,2026-08,9 | NY,2026-09,6 | NY,2026-10,10 | TX,2026-08,7 | TX,2026-09,5 | TX,2026-10,2 | 

[step 2] model asked for calculator({'expression': '(5 - 7) / 7 * 100'})
  result: -28.5714

[step 3] model asked for calculator({'expression': '(2 - 5) / 5 * 100'})
  result: -60.0


'Stopped: hit the step limit without a final answer.'

**Notice in the last run:** with `max_steps=3` the agent may stop before finishing. The step cap trades completeness for cost and safety. In production you log these cases and tune the cap.

## 9. Check the answers yourself (agents must be evaluated)

An agent can write a wrong query and still sound confident. Verify one answer with your own SQL, the same way you checked RAG answers against the source.

In [9]:
print(run_sql("SELECT ROUND(AVG(cost_usd), 2) AS avg_cost FROM claims WHERE state='TX' AND claim_type='auto'"))
print(run_sql("SELECT ROUND(100.0 * AVG(has_attorney), 1) AS pct_attorney FROM claims WHERE claim_type='auto'"))

avg_cost
5808.78

pct_attorney
22.1



## 10. What we just learned

- An agent is an **LLM in a loop** that chooses tools; your code executes them and returns results.
- The model only **requests** tool calls; **guardrails** (SELECT-only, LIMIT, step cap) belong in code.
- The **message history** is the agent's memory within a task.
- Returning **errors as text** lets the agent self-correct.
- Agents must be **evaluated** against known answers, just like RAG (task success, correct tool calls, steps, cost).
- Frameworks like **LangGraph** wrap exactly this loop and add state, branching, retries and tracing.
